[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/32.%20The%20Inter-Terminal%20Transfer%20Optimization%20Problem/P32-Tier-6_executed.ipynb)

---


# 32. The Inter-Terminal Transfer Optimization Problem

## Tier 6: Autonomous Self-Optimizing Ecosystem (Multi-Agent System)

### Goal

Build and inspect a discrete-time, message-passing multi-agent simulation for inter-terminal container transfers. Vehicle agents submit bids, terminal agents administer congestion-triggered auctions, and a coordinator relays announcements and bids. The notebook demonstrates how local decisions and communication can coordinate operations; it does not establish that the system is globally optimal or that an equilibrium is reached.

### Inputs, outputs, and scope

The example uses five terminals, eight vehicles with different capacities and costs, a terminal-to-terminal distance matrix, and randomly generated container arrivals. The simulation produces time-series system metrics, agent-level performance summaries, and plots. No random seed is set, so individual runs can differ.

The model abstracts travel and handling into distances, costs, capacities, queues, and discrete simulation steps. It does not model road networks, travel-time uncertainty, container release schedules, vehicle unloading, strategic misreporting, or detailed terminal operations. The coordinator relays messages and computes simple market indicators; it does not centrally solve an optimization problem. Bids use a lowest-price rule, not a proof of a market equilibrium.

### Concepts and modeling rules

A **multi-agent system** represents an environment as autonomous entities that maintain local state and interact through messages. A **BDI architecture** organizes an agent's state into beliefs (information it holds), desires (goals), and intentions (goals it has selected to pursue). This example provides those structures, while the operational behavior is primarily implemented by explicit message handlers and agent actions.

An **auction** is a local allocation mechanism: eligible vehicles submit prices for a container transfer, and the terminal awards the transfer to the lowest submitted price when the auction closes. An auction is opened only when the terminal's congestion exceeds the stated threshold. The simulation is discrete-time: messages created during an agent-action phase enter a queue and are delivered in a later simulation step.

For a vehicle \(v\) considering a container with origin \(o\) and destination \(d\), the modeled route distance includes travel from the vehicle's current position \(p_v\) to \(o\), followed by the transfer from \(o\) to \(d\):

$$
D_v(p_v,o,d) = d_{p_v,o} + d_{o,d}.
$$

The vehicle is eligible when its remaining capacity is sufficient, its fuel level is above the minimum threshold, and it is active. Its bid price is

$$
b_v = D_v(p_v,o,d)c_v(1+m_v)a_v(u_v),
$$

where \(c_v\) is cost per distance unit, \(m_v\) is the minimum profit margin, \(u_v\) is current load divided by capacity, and \(a_v(u_v)\) is the utilization adjustment defined in the vehicle-agent code. A terminal selects the submitted bid with the smallest price. These rules describe the implemented allocation mechanism; they do not guarantee a globally minimum-cost assignment.

### What to inspect

- How local beliefs and state are updated as agents exchange messages.
- How the coordinator distributes auction announcements and forwards bids.
- Which capacity, fuel, and activity checks determine vehicle eligibility.
- How queue congestion triggers auctions and how the lowest-price bid is selected.
- Whether auctions actually occur in a particular run; the threshold and random arrivals may result in few or no auctions.
- How to interpret the plotted metrics without treating them as evidence of Nash equilibrium, adaptation, or global optimality.

The example is intentionally a small simulation of distributed coordination. It is not a validation study or a benchmark comparison.

In [ ]:
from collections import deque
from dataclasses import dataclass
import random
import time
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8")

In [ ]:
@dataclass
class Message:
    """A message routed between agents during the simulation."""

    sender_id: str
    receiver_id: str
    message_type: str
    content: dict
    timestamp: float
    priority: int = 1


@dataclass
class Auction:
    """An auction for one container transfer."""

    auction_id: str
    resource_type: str
    resource: dict
    bidders: List[str]
    current_bid: Optional[dict]
    deadline: float
    winner: Optional[str] = None
    status: str = "active"


class Agent:
    """Base class providing BDI state and message-handling interfaces."""

    def __init__(self, agent_id: str, agent_type: str):
        self.agent_id = agent_id
        self.agent_type = agent_type

        # These three collections represent the BDI state used by each agent.
        self.beliefs = {}
        self.desires = []
        self.intentions = []

        self.inbox = deque(maxlen=100)
        self.outbox = deque()
        self.message_history = deque(maxlen=1000)
        self.is_active = True
        self.current_task = None
        self.performance_history = deque(maxlen=100)
        self.current_time = 0.0

    def send_message(
        self,
        receiver_id: str,
        message_type: str,
        content: dict,
        priority: int = 1,
    ) -> None:
        """Queue a message so the system can deliver it in a later phase."""
        message = Message(
            sender_id=self.agent_id,
            receiver_id=receiver_id,
            message_type=message_type,
            content=content,
            timestamp=self.current_time,
            priority=priority,
        )
        self.outbox.append(message)
        self.message_history.append(message)

    def receive_message(self, message: Message) -> None:
        """Place a delivered message in this agent's inbox."""
        self.inbox.append(message)
        self.message_history.append(message)

    def process_messages(self) -> None:
        """Handle every message currently waiting in the inbox."""
        while self.inbox:
            self.handle_message(self.inbox.popleft())

    def handle_message(self, message: Message) -> None:
        """Define message handling in a specialized agent class."""
        raise NotImplementedError("Specialized agents must handle messages.")

    def update_beliefs(self, new_beliefs: dict) -> None:
        """Update local information without replacing unrelated beliefs."""
        self.beliefs.update(new_beliefs)

    def add_desire(self, desire: dict) -> None:
        """Record a goal the agent may consider pursuing."""
        self.desires.append(desire)

    def form_intentions(self) -> None:
        """Select desires that the agent considers achievable."""
        self.intentions = [desire for desire in self.desires if self.can_achieve(desire)]

    def can_achieve(self, desire: dict) -> bool:
        """Provide a default feasibility rule for the base agent."""
        return self.is_active

    def act(self) -> None:
        """Allow specialized agents to perform their operational actions."""
        return None

In [ ]:
class VehicleAgent(Agent):
    """Vehicle that evaluates transfer auctions and submits eligible bids."""

    def __init__(
        self,
        agent_id: str,
        capacity: int,
        cost_per_km: float,
        initial_position: str,
        terminals: List[str],
        distances: np.ndarray,
    ):
        super().__init__(agent_id, "vehicle")
        self.capacity = capacity
        self.cost_per_km = cost_per_km
        self.current_position = initial_position
        self.current_load = 0
        self.fuel_level = 100.0
        self.total_distance = 0.0
        self.assigned_containers = []
        self.min_profit_margin = 0.1
        self.risk_tolerance = 0.5
        self.terminals = list(terminals)
        self.distances = np.asarray(distances, dtype=float)
        self.terminal_indices = {
            terminal: index for index, terminal in enumerate(self.terminals)
        }

        self.update_beliefs(
            {
                "position": initial_position,
                "capacity": capacity,
                "current_load": 0,
                "fuel_level": self.fuel_level,
                "market_prices": {},
            }
        )
        self.add_desire({"type": "maximize_profit", "priority": 1})
        self.add_desire(
            {"type": "maintain_utilization", "target": 0.8, "priority": 2}
        )
        self.add_desire({"type": "minimize_empty_runs", "priority": 3})

    def handle_message(self, message: Message) -> None:
        """Respond to market announcements and awarded assignments."""
        if message.message_type == "auction_announcement":
            self.evaluate_auction(message.content)
        elif message.message_type == "assignment":
            self.accept_assignment(message.content)
        elif message.message_type == "market_update":
            self.update_beliefs(message.content)

    def calculate_distance(self, origin: str, destination: str) -> float:
        """Look up a directed terminal-to-terminal distance in the input matrix."""
        if origin not in self.terminal_indices or destination not in self.terminal_indices:
            raise ValueError("Both locations must be present in the terminal list.")
        return float(
            self.distances[
                self.terminal_indices[origin], self.terminal_indices[destination]
            ]
        )

    def evaluate_auction(self, auction_info: dict) -> None:
        """Check feasibility and submit a price for an announced transfer."""
        resource = auction_info["resource"]
        if resource.get("type") != "container":
            return

        weight = resource["weight"]
        eligible = (
            self.is_active
            and self.current_load + weight <= self.capacity
            and self.fuel_level > 20
        )
        if not eligible:
            return

        route_distance = self.calculate_distance(
            self.current_position, resource["origin"]
        ) + self.calculate_distance(resource["origin"], resource["destination"])
        base_cost = route_distance * self.cost_per_km
        utilization = self.current_load / self.capacity
        adjustment = 0.9 if utilization < 0.5 else 1.2 if utilization > 0.8 else 1.0
        bid_price = base_cost * (1 + self.min_profit_margin) * adjustment

        bid = {
            "agent_id": self.agent_id,
            "auction_id": auction_info["auction_id"],
            "auctioneer_id": auction_info["auctioneer_id"],
            "price": bid_price,
            "estimated_time": route_distance / 25,
            "confidence": 1.0 - utilization * self.risk_tolerance,
        }
        self.send_message(
            "coordinator_001", "bid_submission", bid, priority=2
        )

    def accept_assignment(self, assignment: dict) -> None:
        """Record an awarded transfer and update vehicle state."""
        container = assignment["container"]
        distance = self.calculate_distance(
            self.current_position, container["origin"]
        ) + self.calculate_distance(container["origin"], container["destination"])

        self.assigned_containers.append(container)
        self.current_load += container["weight"]
        self.current_position = container["destination"]
        self.total_distance += distance
        self.fuel_level = max(0.0, self.fuel_level - distance * 0.1)
        self.update_beliefs(
            {
                "current_load": self.current_load,
                "position": self.current_position,
                "total_distance": self.total_distance,
                "fuel_level": self.fuel_level,
                "assigned_containers": len(self.assigned_containers),
            }
        )

        actual_cost = distance * self.cost_per_km
        self.performance_history.append(
            {
                "timestamp": self.current_time,
                "action": "assignment",
                "profit": assignment["price"] - actual_cost,
                "utilization": self.current_load / self.capacity,
            }
        )


class TerminalAgent(Agent):
    """Terminal that tracks a queue and runs congestion-triggered auctions."""

    def __init__(self, agent_id: str, terminal_id: str, capacity: int):
        super().__init__(agent_id, "terminal")
        self.terminal_id = terminal_id
        self.capacity = capacity
        self.current_containers = []
        self.processing_rate = 10
        self.processing_credit = 0.0
        self.queue_length = 0
        self.congestion_level = 0.0
        self.active_auctions = {}
        self.local_market_prices = {}
        self.auctions_started = 0
        self.auctions_awarded = 0
        self.auctions_without_bids = 0
        self.time_step = 60

        self.update_beliefs(
            {
                "terminal_id": terminal_id,
                "capacity": capacity,
                "current_containers": 0,
                "processing_rate": self.processing_rate,
                "congestion_level": 0.0,
            }
        )
        self.add_desire({"type": "minimize_congestion", "priority": 1})
        self.add_desire({"type": "maximize_throughput", "priority": 2})
        self.add_desire({"type": "balance_workload", "priority": 3})

    def handle_message(self, message: Message) -> None:
        """Process arrivals, bids, and coordinator requests."""
        if message.message_type == "container_arrival":
            self.process_container_arrival(message.content)
        elif message.message_type == "bid_submission":
            self.process_bid(message.content)
        elif message.message_type == "market_request":
            self.provide_market_update(message.content)

    def _update_queue_state(self) -> None:
        self.queue_length = len(self.current_containers)
        self.congestion_level = min(1.0, self.queue_length / self.capacity)
        self.update_beliefs(
            {
                "current_containers": self.queue_length,
                "queue_length": self.queue_length,
                "congestion_level": self.congestion_level,
            }
        )

    def process_container_arrival(self, container_info: dict) -> None:
        """Add a container to the queue and check the auction threshold."""
        container = container_info["container"]
        self.current_containers.append(container)
        self._update_queue_state()
        if self.congestion_level > 0.7:
            self.create_container_auction(container)

    def create_container_auction(self, container: dict) -> None:
        """Open an auction and announce it through the coordinator."""
        auction_id = f"auction_{self.terminal_id}_{container['id']}"
        if auction_id in self.active_auctions:
            return

        auction = Auction(
            auction_id=auction_id,
            resource_type="container",
            resource=container,
            bidders=[],
            current_bid=None,
            deadline=self.current_time + 3 * self.time_step,
        )
        self.active_auctions[auction_id] = auction
        self.auctions_started += 1
        self.send_message(
            "coordinator_001",
            "auction_announcement",
            {
                "auction_id": auction_id,
                "auctioneer_id": self.agent_id,
                "resource": container,
                "deadline": auction.deadline,
            },
            priority=3,
        )

    def process_bid(self, bid_info: dict) -> None:
        """Keep the lowest valid bid received before the auction deadline."""
        auction = self.active_auctions.get(bid_info["auction_id"])
        if auction is None or self.current_time > auction.deadline:
            return
        if bid_info["auctioneer_id"] != self.agent_id:
            return

        auction.bidders.append(bid_info["agent_id"])
        if (
            auction.current_bid is None
            or bid_info["price"] < auction.current_bid["price"]
        ):
            auction.current_bid = bid_info
            self.local_market_prices[auction.resource["id"]] = bid_info["price"]

    def close_auction(self, auction_id: str) -> None:
        """Award the transfer to the lowest bidder, or record no award."""
        auction = self.active_auctions.get(auction_id)
        if auction is None:
            return

        if auction.current_bid is not None:
            winner_id = auction.current_bid["agent_id"]
            auction.winner = winner_id
            assignment = {
                "container": auction.resource,
                "price": auction.current_bid["price"],
                "winner": winner_id,
            }
            self.send_message(winner_id, "assignment", assignment, priority=3)
            self.current_containers = [
                container
                for container in self.current_containers
                if container["id"] != auction.resource["id"]
            ]
            self.auctions_awarded += 1
            self.performance_history.append(
                {
                    "timestamp": self.current_time,
                    "action": "auction_awarded",
                    "price": auction.current_bid["price"],
                    "winner": winner_id,
                    "congestion_level": self.congestion_level,
                }
            )
        else:
            self.auctions_without_bids += 1
            self.performance_history.append(
                {
                    "timestamp": self.current_time,
                    "action": "auction_without_bids",
                    "congestion_level": self.congestion_level,
                }
            )

        auction.status = "closed"
        del self.active_auctions[auction_id]
        self._update_queue_state()

    def provide_market_update(self, request: dict) -> None:
        """Send queue information to the requesting coordinator."""
        self.send_message(
            request["requester_id"],
            "market_update",
            {
                "terminal_id": self.terminal_id,
                "queue_length": self.queue_length,
                "congestion_level": self.congestion_level,
            },
            priority=1,
        )

    def act(self) -> None:
        """Process queue capacity over time and close expired auctions."""
        self.processing_credit += self.processing_rate * self.time_step / 3600
        processed = min(int(self.processing_credit), len(self.current_containers))
        if processed:
            del self.current_containers[:processed]
            self.processing_credit -= processed
            self._update_queue_state()

        expired = [
            auction_id
            for auction_id, auction in self.active_auctions.items()
            if self.current_time >= auction.deadline
        ]
        for auction_id in expired:
            self.close_auction(auction_id)


class CoordinatorAgent(Agent):
    """Relay auction messages and maintain simple market indicators."""

    def __init__(self, agent_id: str):
        super().__init__(agent_id, "coordinator")
        self.registered_agents = {}
        self.market_state = {
            "total_containers": 0,
            "active_vehicles": 0,
            "average_prices": {},
            "market_efficiency": 0.0,
        }
        self.update_beliefs(
            {"registered_agents": self.registered_agents, "market_state": self.market_state}
        )
        self.add_desire({"type": "market_efficiency", "priority": 1})
        self.add_desire({"type": "price_stability", "priority": 2})
        self.add_desire({"type": "fair_allocation", "priority": 3})

    def handle_message(self, message: Message) -> None:
        """Route registration, auction, bid, and market messages."""
        if message.message_type == "registration":
            self.register_agent(message.content)
        elif message.message_type == "auction_announcement":
            self.distribute_auction(message.content)
        elif message.message_type == "bid_submission":
            self.forward_bid(message.content)
        elif message.message_type == "market_update":
            self.update_market_state(message.content)

    def register_agent(self, agent_info: dict) -> None:
        """Store an agent's type and advertised capabilities."""
        agent_id = agent_info["agent_id"]
        agent_type = agent_info["agent_type"]
        is_new_vehicle = (
            agent_id not in self.registered_agents and agent_type == "vehicle"
        )
        self.registered_agents[agent_id] = {
            "type": agent_type,
            "capabilities": agent_info.get("capabilities", {}),
            "status": "active",
        }
        if is_new_vehicle:
            self.market_state["active_vehicles"] += 1
        self.update_beliefs(
            {"registered_agents": self.registered_agents, "market_state": self.market_state}
        )

    def distribute_auction(self, auction_info: dict) -> None:
        """Send an announcement to registered active vehicle agents."""
        for agent_id, agent_info in self.registered_agents.items():
            if agent_info["type"] == "vehicle" and agent_info["status"] == "active":
                self.send_message(
                    agent_id, "auction_announcement", auction_info, priority=2
                )

    def forward_bid(self, bid_info: dict) -> None:
        """Forward a vehicle bid to the terminal running that auction."""
        auctioneer_id = bid_info.get("auctioneer_id")
        if auctioneer_id in self.registered_agents:
            self.send_message(
                auctioneer_id, "bid_submission", bid_info, priority=2
            )

    def update_market_state(self, market_info: dict) -> None:
        """Aggregate reported terminal queues and calculate a ratio indicator."""
        terminal_id = market_info.get("terminal_id")
        if terminal_id is not None:
            self.market_state[f"terminal_{terminal_id}"] = market_info

        total_containers = sum(
            value.get("queue_length", 0)
            for key, value in self.market_state.items()
            if key.startswith("terminal_")
        )
        total_vehicle_capacity = sum(
            agent["capabilities"].get("capacity", 0)
            for agent in self.registered_agents.values()
            if agent["type"] == "vehicle"
        )
        self.market_state["total_containers"] = total_containers
        if total_vehicle_capacity:
            self.market_state["market_efficiency"] = (
                total_containers / total_vehicle_capacity
            )
        self.update_beliefs({"market_state": self.market_state})

    def act(self) -> None:
        """Request terminal updates; the coordinator does not solve an assignment model."""
        for agent_id, agent_info in self.registered_agents.items():
            if agent_info["type"] == "terminal":
                self.send_message(
                    agent_id,
                    "market_request",
                    {"requester_id": self.agent_id},
                    priority=1,
                )

## Simulation design

The system below creates one coordinator, one terminal agent per terminal, and one vehicle agent per vehicle. Each simulation step advances the clock, delivers messages queued in the preceding step, lets agents process messages and act, then records system metrics. This ordering makes message passing explicit: sending a message does not cause the recipient to act immediately.

The terminal processing rate is measured in containers per hour. A fractional processing credit carries work forward between one-minute steps, so a rate below one container per step is not silently rounded to zero. Auctions remain open for three simulation intervals, giving announcements and bids time to pass through the coordinator.

In [ ]:
class InterTerminalMultiAgentSystem:
    """Discrete-time simulation of terminals, vehicles, and a coordinator."""

    def __init__(self, terminals, vehicles, distances):
        self.terminals = list(terminals)
        self.vehicles = list(vehicles)
        self.distances = np.asarray(distances, dtype=float)
        if self.distances.shape != (len(self.terminals), len(self.terminals)):
            raise ValueError("The distance matrix must match the terminal list.")
        if len(self.terminals) < 2:
            raise ValueError("At least two terminals are required for transfers.")

        self.agents = {}
        self.message_queue = deque()
        self.current_time = 0
        self.simulation_step = 0
        self.system_metrics = deque(maxlen=1000)
        self.auction_history = deque(maxlen=500)
        self.next_container_id = 1
        self._initialize_agents()
        self._initialize_containers()

    def _initialize_agents(self):
        """Create agents and queue their registration messages."""
        coordinator = CoordinatorAgent("coordinator_001")
        self.agents[coordinator.agent_id] = coordinator

        for terminal in self.terminals:
            terminal_agent = TerminalAgent(
                agent_id=f"terminal_{terminal}",
                terminal_id=terminal,
                capacity=50,
            )
            self.agents[terminal_agent.agent_id] = terminal_agent

        for vehicle in self.vehicles:
            vehicle_agent = VehicleAgent(
                agent_id=f"vehicle_{vehicle.id}",
                capacity=vehicle.capacity,
                cost_per_km=vehicle.cost_per_km,
                initial_position=random.choice(self.terminals),
                terminals=self.terminals,
                distances=self.distances,
            )
            self.agents[vehicle_agent.agent_id] = vehicle_agent

        for agent in self.agents.values():
            if agent.agent_type == "coordinator":
                continue
            registration = {
                "agent_id": agent.agent_id,
                "agent_type": agent.agent_type,
                "capabilities": self._get_agent_capabilities(agent),
            }
            agent.send_message(
                "coordinator_001", "registration", registration, priority=3
            )

    def _get_agent_capabilities(self, agent):
        """Return the fields the coordinator needs for registration."""
        if agent.agent_type == "vehicle":
            return {
                "capacity": agent.capacity,
                "cost_per_km": agent.cost_per_km,
                "current_position": agent.current_position,
                "current_load": agent.current_load,
                "fuel_level": agent.fuel_level,
            }
        if agent.agent_type == "terminal":
            return {
                "terminal_id": agent.terminal_id,
                "capacity": agent.capacity,
                "processing_rate": agent.processing_rate,
                "queue_length": agent.queue_length,
                "congestion_level": agent.congestion_level,
            }
        return {}

    def _make_container(self, container_id):
        """Create a transfer whose destination differs from its origin."""
        origin = random.choice(self.terminals)
        destinations = [terminal for terminal in self.terminals if terminal != origin]
        destination = random.choice(destinations)
        return {
            "id": container_id,
            "origin": origin,
            "destination": destination,
            "weight": random.choice([1, 2, 3]),
            "priority": random.choice(["low", "medium", "high"]),
            "ready_time": self.current_time,
            "status": "waiting",
            "type": "container",
        }

    def _send_container_to_terminal(self, container):
        """Queue an arrival at the terminal named as the container's origin."""
        terminal_agent = self.agents[f"terminal_{container['origin']}"]
        terminal_agent.send_message(
            terminal_agent.agent_id,
            "container_arrival",
            {"container": container},
            priority=2,
        )

    def _initialize_containers(self):
        """Generate the initial set of twenty container arrivals."""
        for _ in range(20):
            container = self._make_container(self.next_container_id)
            self.next_container_id += 1
            self._send_container_to_terminal(container)

    def simulate_step(self, time_step=60):
        """Advance simulated time, deliver messages, act, and collect metrics."""
        if time_step <= 0:
            raise ValueError("time_step must be positive.")
        self.current_time += time_step
        self.simulation_step += 1

        for agent in self.agents.values():
            agent.current_time = self.current_time
            if isinstance(agent, TerminalAgent):
                agent.time_step = time_step

        self._process_messages()
        self._execute_agent_actions()

        if self.simulation_step % 10 == 0:
            self._generate_new_containers()

        self._collect_system_metrics()
        return self._get_system_state()

    def _process_messages(self):
        """Deliver messages that were queued during an earlier action phase."""
        while self.message_queue:
            message = self.message_queue.popleft()
            recipient = self.agents.get(message.receiver_id)
            if recipient is not None:
                recipient.receive_message(message)

    def _execute_agent_actions(self):
        """Let each active agent process its inbox and then perform its action."""
        for agent in self.agents.values():
            if not agent.is_active:
                continue
            agent.process_messages()
            agent.form_intentions()
            agent.act()
            while agent.outbox:
                self.message_queue.append(agent.outbox.popleft())

    def _generate_new_containers(self):
        """Generate one to three arrivals every ten simulation steps."""
        for _ in range(random.randint(1, 3)):
            container = self._make_container(self.next_container_id)
            self.next_container_id += 1
            self._send_container_to_terminal(container)

    def _collect_system_metrics(self):
        """Record queues, vehicle loads, and observed auction outcomes."""
        terminals = [
            agent for agent in self.agents.values()
            if isinstance(agent, TerminalAgent)
        ]
        vehicles = [
            agent for agent in self.agents.values()
            if isinstance(agent, VehicleAgent)
        ]
        total_containers = sum(len(agent.current_containers) for agent in terminals)
        total_capacity = sum(agent.capacity for agent in vehicles)
        total_load = sum(agent.current_load for agent in vehicles)
        active_auctions = sum(len(agent.active_auctions) for agent in terminals)
        auctions_started = sum(agent.auctions_started for agent in terminals)
        auctions_awarded = sum(agent.auctions_awarded for agent in terminals)
        auction_success_rate = (
            auctions_awarded / auctions_started if auctions_started else 0.0
        )
        system_efficiency = total_load / total_capacity if total_capacity else 0.0

        metrics = {
            "timestamp": self.current_time,
            "simulation_step": self.simulation_step,
            "total_containers": total_containers,
            "total_capacity": total_capacity,
            "total_load": total_load,
            "system_efficiency": system_efficiency,
            "active_auctions": active_auctions,
            "auctions_started": auctions_started,
            "auctions_awarded": auctions_awarded,
            "auction_success_rate": auction_success_rate,
            "active_agents": sum(agent.is_active for agent in self.agents.values()),
        }
        self.system_metrics.append(metrics)

    def _get_system_state(self):
        """Return a snapshot suitable for progress reporting and analysis."""
        return {
            "current_time": self.current_time,
            "simulation_step": self.simulation_step,
            "agents": {
                agent_id: {
                    "type": agent.agent_type,
                    "is_active": agent.is_active,
                    "beliefs": agent.beliefs,
                    "performance_history": list(agent.performance_history),
                }
                for agent_id, agent in self.agents.items()
            },
            "metrics": dict(self.system_metrics[-1]) if self.system_metrics else {},
        }

    def run_simulation(self, num_steps=100):
        """Run the simulation and return aggregated observed results."""
        if num_steps <= 0:
            raise ValueError("num_steps must be positive.")

        vehicle_count = sum(
            agent.agent_type == "vehicle" for agent in self.agents.values()
        )
        terminal_count = sum(
            agent.agent_type == "terminal" for agent in self.agents.values()
        )
        print("=== MULTI-AGENT SYSTEM SIMULATION ===")
        print(
            f"Agents: {len(self.agents)} "
            f"(Vehicles: {vehicle_count}, Terminals: {terminal_count})"
        )
        print(f"Simulation steps: {num_steps}")
        print("Time step: 60 seconds per step")

        start_time = time.time()
        for step in range(num_steps):
            state = self.simulate_step()
            if step % 10 == 0 or step == num_steps - 1:
                metrics = state["metrics"]
                print(
                    f"Step {step + 1:3d}: "
                    f"Containers: {metrics['total_containers']:3d}, "
                    f"Load: {metrics['total_load']:3d}/"
                    f"{metrics['total_capacity']:3d} "
                    f"({metrics['system_efficiency']:.1%}), "
                    f"Active auctions: {metrics['active_auctions']:2d}, "
                    f"Award rate: {metrics['auction_success_rate']:.1%}"
                )

        elapsed = time.time() - start_time
        print(f"\nSimulation completed in {elapsed:.2f} seconds")
        return self._analyze_results()

    def _analyze_results(self):
        """Summarize recorded metrics and agent performance histories."""
        if not self.system_metrics:
            return {}

        metrics = list(self.system_metrics)
        final_metrics = dict(metrics[-1])
        averages = {
            "efficiency": float(np.mean([item["system_efficiency"] for item in metrics])),
            "success_rate": float(
                np.mean([item["auction_success_rate"] for item in metrics])
            ),
            "containers": float(np.mean([item["total_containers"] for item in metrics])),
        }

        vehicle_performance = []
        terminal_performance = []
        for agent in self.agents.values():
            if isinstance(agent, VehicleAgent) and agent.performance_history:
                profits = [
                    item["profit"]
                    for item in agent.performance_history
                    if "profit" in item
                ]
                if profits:
                    vehicle_performance.append(
                        {
                            "agent_id": agent.agent_id,
                            "avg_profit": float(np.mean(profits)),
                            "total_assignments": len(agent.assigned_containers),
                            "utilization": agent.current_load / agent.capacity,
                        }
                    )
            elif isinstance(agent, TerminalAgent):
                awarded = [
                    item
                    for item in agent.performance_history
                    if item["action"] == "auction_awarded"
                ]
                if awarded:
                    terminal_performance.append(
                        {
                            "agent_id": agent.agent_id,
                            "avg_price": float(np.mean([item["price"] for item in awarded])),
                            "total_auctions": len(awarded),
                            "avg_congestion": float(
                                np.mean([item["congestion_level"] for item in awarded])
                            ),
                        }
                    )

        return {
            "final_metrics": final_metrics,
            "averages": averages,
            "vehicle_performance": vehicle_performance,
            "terminal_performance": terminal_performance,
            "simulation_steps": len(metrics),
            "metrics": metrics,
        }

## Configure and initialize the example

The distance matrix is indexed in the same order as the terminal list. Its entries are the modeled distances used both for vehicle bids and for the cost recorded after an assignment. A vehicle's estimated route includes the distance from its current location to the container origin as well as the origin-to-destination transfer.

In [ ]:
def create_multi_agent_system():
    """Create the example terminals, vehicles, and distance matrix."""
    terminals = ["A", "B", "C", "D", "E"]
    vehicles = [
        type("Vehicle", (), {"id": 1, "capacity": 3, "cost_per_km": 10}),
        type("Vehicle", (), {"id": 2, "capacity": 2, "cost_per_km": 12}),
        type("Vehicle", (), {"id": 3, "capacity": 4, "cost_per_km": 8}),
        type("Vehicle", (), {"id": 4, "capacity": 3, "cost_per_km": 11}),
        type("Vehicle", (), {"id": 5, "capacity": 2, "cost_per_km": 13}),
        type("Vehicle", (), {"id": 6, "capacity": 5, "cost_per_km": 9}),
        type("Vehicle", (), {"id": 7, "capacity": 3, "cost_per_km": 10}),
        type("Vehicle", (), {"id": 8, "capacity": 2, "cost_per_km": 14}),
    ]
    distance_matrix = np.array(
        [
            [0, 3, 5, 7, 6],
            [3, 0, 4, 6, 5],
            [5, 4, 0, 4, 3],
            [7, 6, 4, 0, 3],
            [6, 5, 3, 3, 0],
        ],
        dtype=float,
    )
    return InterTerminalMultiAgentSystem(terminals, vehicles, distance_matrix)


mas = create_multi_agent_system()
print("=== MULTI-AGENT SYSTEM INITIALIZATION ===")
print(f"Terminals: {mas.terminals}")
print(f"Vehicles: {len(mas.vehicles)}")
print(f"Total agents: {len(mas.agents)}")
print(
    "Agent types: "
    f"{sorted({agent.agent_type for agent in mas.agents.values()})}"
)
print("Coordinator: coordinator_001")

## Run the simulation and inspect the results

The auction award rate is calculated as the cumulative number of awarded auctions divided by the cumulative number of auctions opened. It is zero when no auction has been opened. That convention makes the metric defined for every step, but a zero value does not distinguish between no auctions and auctions that received no bids; use the separate auction counts to interpret it.

In [ ]:
results = mas.run_simulation(num_steps=50)

print("\n=== MULTI-AGENT SYSTEM RESULTS ===")
final_metrics = results["final_metrics"]
averages = results["averages"]
print("\nFinal system metrics:")
print(f"  Containers in queues: {final_metrics['total_containers']}")
print(f"  Mean vehicle-load ratio: {averages['efficiency']:.1%}")
print(f"  Mean cumulative auction award rate: {averages['success_rate']:.1%}")
print(f"  Mean containers in queues: {averages['containers']:.1f}")
print(f"  Auctions opened: {final_metrics['auctions_started']}")
print(f"  Auctions awarded: {final_metrics['auctions_awarded']}")
print(f"  Active agents: {final_metrics['active_agents']}")

print("\nVehicle-agent performance (agents with awarded assignments):")
if results["vehicle_performance"]:
    for performance in results["vehicle_performance"]:
        print(f"  {performance['agent_id']}:")
        print(f"    Mean recorded profit: {performance['avg_profit']:.2f} cost units")
        print(f"    Total assignments: {performance['total_assignments']}")
        print(f"    Load-to-capacity ratio: {performance['utilization']:.1%}")
else:
    print("  No vehicle assignments were recorded.")

print("\nTerminal-agent performance (terminals with awarded auctions):")
if results["terminal_performance"]:
    for performance in results["terminal_performance"]:
        print(f"  {performance['agent_id']}:")
        print(f"    Mean awarded price: {performance['avg_price']:.2f} cost units")
        print(f"    Auctions awarded: {performance['total_auctions']}")
        print(f"    Mean congestion at award: {performance['avg_congestion']:.1%}")
else:
    print("  No auction awards were recorded.")

print(f"\nRecorded simulation steps: {results['simulation_steps']}")

## Visualize simulation metrics

Each panel shows a distinct view of the recorded run. The load-to-capacity ratio is a utilization indicator for the simulated vehicles, not a measure of global optimality. Auction award rate is cumulative and should be read alongside the cumulative auction counts. The equilibrium reference in the final panel is only a descriptive late-run average; it is not evidence that the agents have reached a game-theoretic equilibrium.

In [ ]:
def visualize_multi_agent_results(results):
    """Plot time-series and agent-level metrics from one simulation run."""
    metrics = results["metrics"]
    if not metrics:
        raise ValueError("The results contain no recorded simulation metrics.")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Multi-Agent System Simulation Metrics", fontsize=16, fontweight="bold")

    steps = [item["simulation_step"] for item in metrics]
    efficiency = [100 * item["system_efficiency"] for item in metrics]
    award_rate = [100 * item["auction_success_rate"] for item in metrics]

    ax = axes[0, 0]
    ax.plot(steps, efficiency, color="tab:blue", linewidth=2, label="Vehicle load / capacity")
    ax.plot(steps, award_rate, color="tab:red", linewidth=2, label="Cumulative auction award rate")
    ax.set_xlabel("Simulation step")
    ax.set_ylabel("Percentage (%)")
    ax.set_title("Load Ratio and Auction Outcomes")
    ax.set_ylim(0, 100)
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    queued = [item["total_containers"] for item in metrics]
    capacity = [item["total_capacity"] for item in metrics]
    load = [item["total_load"] for item in metrics]
    ax.plot(steps, queued, marker="o", markersize=3, label="Containers in terminal queues")
    ax.plot(steps, load, label="Assigned vehicle load")
    ax.plot(steps, capacity, linestyle="--", label="Total vehicle capacity")
    ax.set_xlabel("Simulation step")
    ax.set_ylabel("Container-weight units")
    ax.set_title("Queues, Assigned Load, and Capacity")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    vehicle_data = results["vehicle_performance"]
    if vehicle_data:
        labels = [item["agent_id"] for item in vehicle_data]
        profits = [item["avg_profit"] for item in vehicle_data]
        positions = np.arange(len(labels))
        bars = ax.bar(positions, profits, color="tab:blue", alpha=0.8, label="Mean recorded profit")
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=40, ha="right")
        ax.set_ylabel("Profit (cost units)")
        ax.set_title("Vehicle-Agent Mean Recorded Profit")
        ax.grid(True, axis="y", alpha=0.3)
        utilization_axis = ax.twinx()
        utilizations = [100 * item["utilization"] for item in vehicle_data]
        utilization_axis.plot(
            positions,
            utilizations,
            color="tab:red",
            marker="o",
            label="Load / capacity",
        )
        utilization_axis.set_ylabel("Load / capacity (%)")
        bar_handles, bar_labels = ax.get_legend_handles_labels()
        line_handles, line_labels = utilization_axis.get_legend_handles_labels()
        ax.legend(bar_handles + line_handles, bar_labels + line_labels, loc="best")
    else:
        ax.text(0.5, 0.5, "No vehicle assignments were recorded.", ha="center", va="center")
        ax.set_axis_off()

    ax = axes[1, 1]
    efficiency_history = [item["system_efficiency"] for item in metrics]
    window_size = 5
    moving_average = [
        float(np.mean(efficiency_history[max(0, index - window_size + 1): index + 1]))
        for index in range(len(efficiency_history))
    ]
    ax.plot(steps, efficiency_history, alpha=0.4, color="tab:blue", label="Step load ratio")
    ax.plot(
        steps,
        moving_average,
        color="tab:red",
        linewidth=2,
        label=f"Moving average ({window_size} steps)",
    )
    late_run_mean = float(np.mean(moving_average[-10:]))
    ax.axhline(
        late_run_mean,
        color="tab:green",
        linestyle="--",
        label=f"Mean of final moving-average values ({late_run_mean:.2f})",
    )
    ax.set_xlabel("Simulation step")
    ax.set_ylabel("Vehicle load / capacity (ratio)")
    ax.set_title("Load-Ratio Trend (Descriptive)")
    ax.set_ylim(bottom=0)
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_multi_agent_results(results)

## Summary and interpretation

This implementation demonstrates a specific distributed message-passing workflow:

1. Terminal agents add arriving containers to local queues.
2. A terminal opens an auction when its congestion ratio exceeds the model threshold.
3. The coordinator forwards the announcement to registered active vehicles.
4. Eligible vehicles estimate route cost and submit bids through the coordinator.
5. The terminal retains the lowest bid it receives before the deadline and awards the transfer when the auction closes.
6. The simulation records queue, load, and auction metrics for later inspection.

The equations and procedures describe a local auction mechanism, not a centralized optimization model. The coordinator provides communication and aggregation; it does not compute a globally optimal allocation. The BDI fields make beliefs, desires, and intentions explicit, but the example does not implement machine learning or demonstrate that agents adapt their strategies over time.

### Interpreting the reported metrics

- **Vehicle load / capacity** is total assigned container weight divided by total vehicle capacity. It is a utilization indicator under this model's persistent assignment-load abstraction.
- **Auction award rate** is the cumulative fraction of opened auctions that receive an award. Inspect the auction counts as well, since no auctions and auctions without bids have different meanings.
- **Containers in terminal queues** counts containers still waiting at terminals. The terminal processing rate and discrete time step affect this quantity.
- **Vehicle profit** is the awarded bid price minus the modeled route cost. It is a simplified accounting measure, not a complete measure of economic welfare.
- **Late-run mean** is a descriptive statistic for the plotted load ratio. It does not establish convergence or Nash equilibrium.

Random initial positions and arrivals mean that a run's numerical results are not guaranteed to repeat. The congestion threshold may also prevent auctions from occurring in some runs. An empty performance table is therefore a possible result, not evidence of a failed simulation.

### Modeling limitations

The example does not unload vehicle capacity after an assignment, model detailed job completion, or enforce a system-wide vehicle schedule. Terminal processing and auction awards both affect queue counts, so queue behavior should be interpreted within these simplified rules. The market's lowest-price rule considers only submitted bids; it does not prove global optimality, fairness, incentive compatibility, or resilience to agent failures. No statistical comparison, target achievement, agent-satisfaction measure, or equilibrium test is performed.

Natural extensions include adding explicit transfer completion and vehicle unloading, recording rejected or expired auctions separately, and testing additional operating conditions while retaining the same message-passing and auction-based solution approach.